In [0]:
dbutils.widgets.text("landing_timestamp", "")
dbutils.widgets.text("table_name", "")
dbutils.widgets.text("merge_keys", "")
dbutils.widgets.text("load_type", "")

landing_timestamp = dbutils.widgets.get("landing_timestamp")
table_name        = dbutils.widgets.get("table_name")
merge_keys = dbutils.widgets.get("merge_keys")
load_type        = dbutils.widgets.get("load_type")

In [0]:
from pyspark.sql import functions as F

bronze_df = (
    spark.read.table(f"adb_caresync.bronze.{table_name}")
    .filter(
        F.col("landing_timestamp") ==
        F.to_timestamp(
            F.lit(landing_timestamp),
            "yyyy-MM-dd_HH:mm:ss"
        )
    )
)


In [0]:
from pyspark.sql import functions as F

updated_df = (
    bronze_df
    .withColumn("insert_timestamp", F.current_timestamp())
    .withColumn("update_timestamp", F.current_timestamp())
)


In [0]:
from delta.tables import DeltaTable

print("=" * 60)
print("  BRONZE → SILVER WRITE STEP")
print("=" * 60)
print(f"  table_name       : {table_name}")
print(f"  load_type        : {load_type}")
print(f"  merge_keys       : {merge_keys if merge_keys else '(not provided)'}")
print(f"  landing_timestamp: {landing_timestamp}")
print("-" * 60)

target_table = f"adb_caresync.silver.{table_name}"
print(f"[INFO] Target table resolved to: '{target_table}'")

# Count source records before writing
source_count = updated_df.count()
print(f"[INFO] Source DataFrame row count: {source_count}")
print(f"[INFO] Source DataFrame columns  : {updated_df.columns}")
print("-" * 60)

if load_type.upper() == "FULL":
    print(f"[INFO] Load type = FULL — entire target table will be overwritten.")
    query = f"updated_df.write.mode('overwrite').saveAsTable('{target_table}')"
    print(f"[QUERY] {query}")
    updated_df.write.mode("overwrite").saveAsTable(target_table)
    print(f"[SUCCESS] FULL overwrite complete on '{target_table}'. Rows written: {source_count}")

elif load_type.upper() == "APPEND":
    print(f"[INFO] Load type = APPEND — new records will be appended to the target table.")
    query = f"updated_df.write.mode('append').saveAsTable('{target_table}')"
    print(f"[QUERY] {query}")
    updated_df.write.mode("append").saveAsTable(target_table)
    print(f"[SUCCESS] APPEND complete on '{target_table}'. Rows appended: {source_count}")

elif load_type.upper() == "MERGE":
    print(f"[INFO] Load type = MERGE — performing upsert into the target table.")

    if not merge_keys:
        raise ValueError("merge_keys must be provided for MERGE load type.")

    keys = [k.strip() for k in merge_keys.split(",")]
    print(f"[INFO] Merge keys parsed: {keys}")

    merge_condition = " AND ".join(
        [f"target.{k} = source.{k}" for k in keys]
    )
    print(f"[QUERY] Merge condition: {merge_condition}")

    update_set = {col: f"source.{col}" for col in updated_df.columns}
    print(f"[QUERY] whenMatchedUpdate set clause:")
    for k, v in update_set.items():
        print(f"         {k} = {v}")
    print(f"[QUERY] whenNotMatchedInsertAll() — all source columns inserted for new records.")

    table_exists = spark.catalog.tableExists(target_table)
    print(f"[INFO] Target table exists: {table_exists}")

    if not table_exists:
        print(f"[INFO] Target table '{target_table}' does not exist — performing initial full load instead of MERGE.")
        query = f"updated_df.write.mode('overwrite').saveAsTable('{target_table}')"
        print(f"[QUERY] {query}")
        updated_df.write.mode("overwrite").saveAsTable(target_table)
        print(f"[SUCCESS] Initial full load complete on '{target_table}'. Rows written: {source_count}")
    else:
        print(f"[INFO] Target table found. Executing MERGE...")
        full_merge_query = (
            f"DeltaTable('{target_table}').alias('target')\n"
            f"  .merge(source, '{merge_condition}')\n"
            f"  .whenMatchedUpdate(set={{...all columns...}})\n"
            f"  .whenNotMatchedInsertAll()\n"
            f"  .execute()"
        )
        print(f"[QUERY] Full MERGE statement:\n{full_merge_query}")

        delta_table = DeltaTable.forName(spark, target_table)
        (
            delta_table.alias("target")
            .merge(updated_df.alias("source"), merge_condition)
            .whenMatchedUpdate(
                set={col: f"source.{col}" for col in updated_df.columns}
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        print(f"[SUCCESS] MERGE complete on '{target_table}' using keys: {keys}. Rows processed from source: {source_count}")

else:
    raise ValueError(
        f"Unsupported load_type: '{load_type}'. Must be one of: FULL, APPEND, MERGE."
    )

print("-" * 60)
print("  WRITE STEP COMPLETE")
print("=" * 60)


In [0]:
record_count = updated_df.count()
dbutils.notebook.exit(str(record_count))